# Results and metrics audit

This notebook inspects the **reported** CSV results committed with this repository. It does not rerun model training and must not be presented as a fresh reproduction of the original experiment.

The main result is the Random Forest operating on the 48-dimensional fusion `[Z || H]`: 32 Autoencoder latent features plus 16 GCN hidden features. The AE+RF baseline is slightly stronger on overall accuracy, F1 and false-negative rate in the currently reported comparison.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
RESULTS = ROOT / "results"
final_metrics = pd.read_csv(RESULTS / "final_model_metrics.csv")
comparison = pd.read_csv(RESULTS / "model_comparison.csv")

display(final_metrics)
display(comparison)

## Recompute selected metrics from the reported confusion matrix

For binary attack detection, treat `Attack` as the positive class. A false negative is an attack flow classified as benign; a false positive is a benign flow classified as an attack.

In [ ]:
row = final_metrics.iloc[0]
tn, fp, fn, tp = (int(row[c]) for c in ["TN", "FP", "FN", "TP"])

accuracy = (tn + tp) / (tn + fp + fn + tp)
precision = tp / (tp + fp)
recall = tp / (tp + fn)
f1 = 2 * precision * recall / (precision + recall)
fpr = fp / (fp + tn)
fnr = fn / (fn + tp)
specificity = tn / (tn + fp)
balanced_accuracy = (recall + specificity) / 2

recomputed = pd.DataFrame({
    "metric": ["accuracy", "attack_precision", "attack_recall", "f1", "fpr", "fnr", "balanced_accuracy"],
    "recomputed": [accuracy, precision, recall, f1, fpr, fnr, balanced_accuracy],
    "stored": [row["accuracy"], row["attack_precision"], row["attack_recall"], row["f1"],
               row["fpr"], row["fnr"], row["balanced_accuracy"]],
})
recomputed["absolute_difference"] = (recomputed["recomputed"] - recomputed["stored"]).abs()
display(recomputed.style.format({"recomputed":"{:.6f}", "stored":"{:.6f}", "absolute_difference":"{:.8f}"}))
print(f"Test flows: {tn + fp + fn + tp:,}")
print(f"Benign test flows: {tn + fp:,}; attack test flows: {fn + tp:,}")

## Compare AE+RF with the proposed fusion

The bars below summarize the already-reported test results. They are not a new experiment. Do not describe AE–GNN–RF as the overall winner: the baseline has slightly higher accuracy and F1 and a slightly lower false-negative rate.

In [ ]:
models = comparison[comparison["model"].isin(["AE+RF", "AE+GNN+RF [Z||H]"])].copy()
labels = ["AE+RF baseline", "AE–GNN–RF [Z || H]"]
models = models.set_index("model").loc[["AE+RF", "AE+GNN+RF [Z||H]"]]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
charts = [
    ("accuracy", "Accuracy", True),
    ("f1", "F1-score", True),
    ("fnr", "False-negative rate", False),
]
for ax, (metric, title, percent) in zip(axes, charts):
    vals = models[metric].astype(float).to_numpy() * 100
    bars = ax.bar(labels, vals)
    ax.set_title(title)
    ax.set_ylabel("%")
    ax.tick_params(axis="x", rotation=25)
    ax.bar_label(bars, fmt="%.2f%%", padding=3)
    if metric == "fnr":
        ax.set_ylim(0, max(vals) * 1.5)
    else:
        ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()

## Correct interpretation for a presentation

- `Z` represents the 32-dimensional Autoencoder latent vector.
- `H` represents the 16-dimensional hidden representation from the GCN.
- `[Z || H]` concatenates both vectors into 48 features for the Random Forest.
- The fusion improves substantially over the RF(H)-only baseline on the listed overall metrics, but remains slightly below AE+RF.
- The next experiment should compare models at a fixed target attack recall, then compare false-positive rate, per-attack misses, and cross-day generalisation.
- Select thresholds on validation data only. Freeze each threshold before evaluating the held-out test set.
- The AE in the reported run was trained on the full training split, which included benign and attack examples; it was not trained exclusively on benign traffic.